# Memory

1. **Short-Term Memory: In-Memory** (RAM-based chat history)
2. **Short-Term Memory: SQL-Based** (SQLite disk-based chat history)
3. **Short-Term Memory: MongoDB-Based** (Document NoSQL-based chat history)
4. **Long-Term Memory: Vector Database** (Semantic facts retriever memory)


### 🛠️ Step 0: Setup LLM & API Keys
We load our environment variables and use `init_chat_model` to instantiate our real LLM. We will use Google's `gemini-2.5-flash` model by default because it offers massive rate limits and superb tool calling/conversational capabilities.

In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

llm = init_chat_model("google_genai:gemini-2.5-flash")
llm

/Users/krishrohilla/Desktop/MindzKonnected/Tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x11aa6c750>, default_metadata=(), model_kwargs={})

## ⚡ 1. Short-Term Memory: In-Memory Chat History
This stores conversation logs in a Python dictionary in your computer's RAM. We will fetch the message list, convert it to LangChain message objects, invoke the LLM, and append the assistant's real response back to memory.

In [5]:
class InMemoryMemory:
    def __init__(self):
        # Stores session_id -> list of message dicts
        self.db = {}
    
    def add_message(self, session_id: str, role: str, content: str):
        if session_id not in self.db:
            self.db[session_id] = []
        self.db[session_id].append({"role": role, "content": content})
        
    def get_messages(self, session_id: str) -> list:
        return self.db.get(session_id, [])
    
    def get_langchain_messages(self, session_id: str) -> list:
        """Convert stored dict messages into LangChain message objects."""
        messages = []
        for msg in self.get_messages(session_id):
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                messages.append(AIMessage(content=msg["content"]))
        return messages

memory = InMemoryMemory()
session_id = "user_123"

memory.add_message(session_id, "user", "Hi, my name is Krish!")

chat_history = memory.get_langchain_messages(session_id)
response_1 = llm.invoke(chat_history)

memory.add_message(session_id, "assistant", response_1.content)

memory.add_message(session_id, "user", "What is my name?")

chat_history = memory.get_langchain_messages(session_id)
response_2 = llm.invoke(chat_history)
memory.add_message(session_id, "assistant", response_2.content)

# Print complete conversation
print("--- InMemory Conversation History ---")
for msg in memory.get_messages(session_id):
    print(f"{msg['role'].upper()}: {msg['content']}")

--- InMemory Conversation History ---
USER: Hi, my name is Krish!
ASSISTANT: Hi Krish! It's great to meet you.

How can I help you today?
USER: What is my name?
ASSISTANT: Your name is Krish!


##  2. Short-Term Memory: SQL-Based Chat History
SQL memory persists conversation logs inside a local database file. Even if the process restarts, the logs are saved. 

In [6]:
import sqlite3

class SQLMemory:
    def __init__(self, db_path="chatbot_memory.db"):
        self.db_path = db_path
        self._create_table()
        
    def _create_table(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT,
                role TEXT,
                content TEXT
            )
        """)
        conn.commit()
        conn.close()
        
    def add_message(self, session_id: str, role: str, content: str):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO messages (session_id, role, content) 
            VALUES (?, ?, ?)
        """, (session_id, role, content))
        conn.commit()
        conn.close()
        
    def get_messages(self, session_id: str) -> list:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            SELECT role, content FROM messages 
            WHERE session_id = ? ORDER BY id ASC
        """, (session_id,))
        rows = cursor.fetchall()
        conn.close()
        return [{"role": row[0], "content": row[1]} for row in rows]
        
    def get_langchain_messages(self, session_id: str) -> list:
        messages = []
        for msg in self.get_messages(session_id):
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                messages.append(AIMessage(content=msg["content"]))
        return messages

sql_memory = SQLMemory(db_path="chatbot_memory.db")
session_id = "user_456"

user_input_1 = "Which city has the Eiffel Tower?"
sql_memory.add_message(session_id, "user", user_input_1)

chat_history = sql_memory.get_langchain_messages(session_id)
response_1 = llm.invoke(chat_history)
sql_memory.add_message(session_id, "assistant", response_1.content)

user_input_2 = "What language is spoken there?"
sql_memory.add_message(session_id, "user", user_input_2)

chat_history = sql_memory.get_langchain_messages(session_id)
response_2 = llm.invoke(chat_history)
sql_memory.add_message(session_id, "assistant", response_2.content)

print("--- SQL Persistent Conversation History ---")
for msg in sql_memory.get_messages(session_id):
    print(f"{msg['role'].upper()}: {msg['content']}")

--- SQL Persistent Conversation History ---
USER: Which city has the Eiffel Tower?
ASSISTANT: The Eiffel Tower is in **Paris**, France.
USER: What language is spoken there?
ASSISTANT: The official language spoken in Paris, France is **French**.


## MongoDB

In [7]:
import mongomock

class MongoDBMemory:
    def __init__(self):
        self.client = mongomock.MongoClient()
        self.db = self.client.chatbot_db
        self.collection = self.db.chat_histories
        
    def add_message(self, session_id: str, role: str, content: str):
        self.collection.update_one(
            {"session_id": session_id},
            {"$push": {"messages": {"role": role, "content": content}}},
            upsert=True
        )
        
    def get_messages(self, session_id: str) -> list:
        doc = self.collection.find_one({"session_id": session_id})
        if doc and "messages" in doc:
            return doc["messages"]
        return []
        
    def get_langchain_messages(self, session_id: str) -> list:
        messages = []
        for msg in self.get_messages(session_id):
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                messages.append(AIMessage(content=msg["content"]))
        return messages

mongo_memory = MongoDBMemory()
session_id = "user_789"

user_input_1 = "I love eating fresh pizza at home."
mongo_memory.add_message(session_id, "user", user_input_1)

chat_history = mongo_memory.get_langchain_messages(session_id)
response_1 = llm.invoke(chat_history)
mongo_memory.add_message(session_id, "assistant", response_1.content)

user_input_2 = "What did I say I like to eat?"
mongo_memory.add_message(session_id, "user", user_input_2)

chat_history = mongo_memory.get_langchain_messages(session_id)
response_2 = llm.invoke(chat_history)
mongo_memory.add_message(session_id, "assistant", response_2.content)

for msg in mongo_memory.get_messages(session_id):
    print(f"{msg['role'].upper()}: {msg['content']}")

USER: I love eating fresh pizza at home.
ASSISTANT: That sounds absolutely delicious! There's nothing quite like fresh pizza, especially in the comfort of your own home.

What kind of fresh pizza do you love to make or order? Are you talking about homemade dough and all, or a fantastic store-bought option you bake yourself?
USER: What did I say I like to eat?
ASSISTANT: You said you love eating **fresh pizza at home**.
